In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath("/Users/charles/Documents/PhD/Analysis/ieeg-pipeline")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from analysis.decoding.loader import iEEGDataLoader
from analysis.decoding.process_features import Features
from analysis.decoding.decoding import Decoding
from analysis.decoding.config import *

In [ ]:
from scipy.stats import ttest_ind

# Load data 

Import for specific subject and check the data to collect



In [ ]:
event_name = "fb"
tmin = -1.5
tmax = 1.5

subjects = [2, 3, 4, 5, 8, 9, 12, 14, 16, 19, 20, 23, 25, 28]

subject = subjects[2]   

power_path = os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "aligned", f"sub-{int(subject):03}_tfr-realign-{event_name}_{tmin}-{tmax}_power.npy")
metadata_path = os.path.join(os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "aligned", f"sub-{int(subject):03}_tfr-realign-{event_name}-{tmin}-{tmax}_metadata.json"))
with open(metadata_path, 'r', encoding='utf-8') as f: 
    metadata = json.load(f)
n_epochs = metadata["n_epochs"]
n_channels = metadata["n_channels"]
event_to_keep = metadata["keep_events"]
power_data = np.load(power_path, mmap_mode='r')

features = Features(subject)

# features.baseline_signal()
# features.extract_power_bands()




In [ ]:
power_path = os.path.join(DATA_DIR, f"sub-{int(subject):03}", "preprocessed", "aligned", f"sub-{int(subject):03}_tfr-realign-{event_name}_{tmin}-{tmax}_power.npy")

features.load_data(phase=False, path=power_path)
# features.prepare_data()

In [ ]:
features.baseline_signal()
features.extract_power_bands(baselined=False)


In [ ]:
anat_df = features.loader.load_anatomy()
test_decoding = Decoding(features.power_bands, features.beh, anat_df)


In [ ]:
test_decoding.create_pipeline(n_folds=5, classification=True)

In [ ]:
# for var in ["choice", "firstswitch", "fb"] :
var = "is_random"
print(f"Decoding variable: {var}")
test_decoding.run_decoding(var, n_jobs = -1, model_type = "hmm")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
# axs = axs.flatten()
# for i, var in enumerate(["choice", "firstswitch", "fb"]) :
#     score = test_decoding.score_global_dict[var]
#     axs[i].plot(np.arange(len(score)), score)
#     axs[i].axhline(0.5, color='black', linestyle='--')
#     axs[i].set_title(f"{var} score time course")
# plt.ylim(0.3, 0.8)
# test_decoding.score_global_dict[var]
_ = test_decoding.plot_tc(var, y = "score", ax=ax, ylim=(0.4, 0.7), save=False, fig=fig)
# test_decoding.plot_multi_tc("", ylim=(0.4, 0.7), save=False)



In [ ]:
# idx_best = np.argmax(test_decoding.score_global_dict["choice"])
var = "is_random"
idx_sd = np.argsort(test_decoding.score_global_dict[var])[-6:]
ordered_regions = anat_df.sort_values(by='region').reset_index()
fig, axs = plt.subplots(3, 2, figsize=(10, 15))
axs = axs.flatten()
for i, idx in enumerate(idx_sd) :
    mat = np.abs(test_decoding.betas_global_dict[var][idx][ordered_regions["chan_idx"].values])
    lim = np.max(np.abs(mat))
    mask = mat <= 0.1
    mat[mask] = 0
    im = axs[i].imshow(mat, aspect='auto', origin='lower', cmap='jet', vmin=0, vmax=lim)
    axs[i].set_yticks(ticks=np.arange(0, len(anat_df)), labels=ordered_regions['region'], fontsize=6)
    axs[i].set_xticks(ticks=np.arange(0, n_frband), labels=FREQUENCY_BANDS.keys(), rotation=45, fontsize=8)
    axs[i].set_title(f"Fold {i+1} - Score: {test_decoding.score_global_dict[var][idx]:.2f}")
    plt.colorbar(im, ax=axs[i])
plt.tight_layout()
# plt.colorbar()

In [ ]:
# for var in ["choice", "firstswitch", "fb"] :
# print(f"Decoding variable: {var}")
var = "is_random"
test_decoding.run_decoding(var, n_jobs = -1, mode = "channel", model_type = "hmm")

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 8))
# _ = test_decoding.plot_heatmap(var, y = "score", save = False, fig=fig, ax=ax)
test_decoding.plot_multi_heatmap("", y = "score", save=False)
# plt.show()

In [ ]:
# idx_best = np.argmax(test_decoding.score_global_dict["choice"])
var = "firstswitch"
idx_sd = np.argsort(test_decoding.score_global_dict[var])[-5:]
ordered_regions = anat_df.sort_values(by='region').reset_index()
fig, axs = plt.subplots(3, 2, figsize=(10, 15))
axs = axs.flatten()
for i, idx in enumerate(idx_sd) :
    mat = test_decoding.betas_channel_dict[var][idx][ordered_regions["chan_idx"].values]
    lim = np.max(np.abs(mat))
    # mask = mat <= 0.2
    # mat[mask] = 0
    im = axs[i].imshow(mat, aspect='auto', origin='lower', cmap='jet', vmin=-lim, vmax=lim)
    axs[i].set_yticks(ticks=np.arange(0, len(anat_df)), labels=ordered_regions['region'], fontsize=6)
    axs[i].set_xticks(ticks=np.arange(0, n_frband), labels=FREQUENCY_BANDS.keys(), rotation=45, fontsize=8)
    axs[i].set_title(f"Fold {i+1} - Score: {test_decoding.score_global_dict[var][idx]:.2f}")
    plt.colorbar(im, ax=axs[i])
plt.tight_layout()
# plt.colorbar()

In [ ]:
features.power_bands

In [ ]:
var = "is_random"
choices = features.beh[var].values

idx_best = np.argmax(test_decoding.score_global_dict[var])
# idx_worst = np.argmin(test_decoding.score_global_dict[var])
mat = np.abs(test_decoding.betas_global_dict[var][idx_best])
# flat_idx = np.argsort(mat)[-5:]
flat_idx = np.argsort(mat.ravel())[-5:]
flat_idx = np.unravel_index(flat_idx, mat.shape)
fig, ax = plt.subplots(5, len(np.unique(choices))+1, figsize=(12, 20))
for j, (channel_idx, band_idx) in enumerate(zip(*flat_idx)):
    diff = []
    for i, choice in enumerate(np.unique(choices)):
        mask = choices == choice
        to_plot = features.power_bands[:, channel_idx, band_idx, :][mask]
        avrg = np.mean(to_plot, axis=0)
        sem = np.std(to_plot, axis=0)
        ax[j, i].plot(to_plot.T, alpha=0.1, lw=0.5, color='gray')
        ax[j, i].fill_between(np.arange(avrg.shape[0]), avrg - sem, avrg + sem, alpha=0.2, color='blue')
        ax[j, i].plot(avrg, lw=2, c='blue')
        ax[j, i].set_title(f"{var}-{choice} | Channel {anat_df['region'][channel_idx]} - Band {list(FREQUENCY_BANDS.keys())[band_idx]}")
        ax[j, i].axvline(idx_best, color='green', linestyle='--', label='Best fold')
        ax[j, i].axvline(1.5*sr_decimated, color='red', linestyle='--', label='fb')
        ax[j, i].legend()
        diff.append(to_plot[:, idx_best])
        # t, p = ttest_ind(diff[0], diff[1])
    ax[j, -1].boxplot(diff)
    ax[j, -1].set_title(f"T-test p-value: {p:.3e}")
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(15, 15), sharex=True)
axs = axs.flatten()
var = "is_random"
fb = features.beh[var].values
for i, ax in enumerate(axs) :
    n_time = test_decoding.n_timepoint
    idx_best = np.argsort(test_decoding.score_global_dict[var])[-i]
    # idx_worst = np.argmin(test_decoding.score_global_dict[var])
    mat = np.abs(test_decoding.betas_global_dict[var][idx_best])
    # flat_idx = np.argsort(mat)[-5:]
    flat_idx = np.argsort(mat.ravel())[-3:]
    flat_idx = np.unravel_index(flat_idx, mat.shape)
    # fig, ax = plt.subplots(figsize=(12, 12))
    for j, (channel_idx, band_idx) in enumerate(zip(*flat_idx)):
        diff_t = np.zeros(n_time)
        # channel_idx, band_idx = flat_idx
        for t in range(n_time):
            pos_idx = fb == 1
            neg_idx = fb == 0
            pos_sig = features.power_bands[:, channel_idx, band_idx, t][pos_idx] 
            neg_sig = features.power_bands[:, channel_idx, band_idx, t][neg_idx]
            # t_, p = ttest_ind(pos_sig, neg_sig)
            diff_t[t] = np.mean(pos_sig) - np.mean(neg_sig)
        ax.plot(diff_t, label=f"Channel {anat_df['region'][channel_idx]} - Band {list(FREQUENCY_BANDS.keys())[band_idx]}")
    ax.axvline(1.5*sr_decimated, color='red', linestyle='--')
    ax.legend()
    ax.set_title(f"best fold {i+1} - Score: {test_decoding.score_global_dict[var][idx_best]:.2f} - Time {idx_best/sr_decimated}")
# ax[j, i].plot(to_plot.T, alpha=0.1, lw=0.5, color='gray')
# ax[j, i].fill_between(np.arange(avrg.shape[0]), avrg - sem, avrg + sem, alpha=0.2, color='blue')
# ax[j, i].plot(avrg, lw=2, c='blue')
# ax[j, i].set_title(f"{var}-{choice} | Channel {anat_df['region'][channel_idx]} - Band {list(FREQUENCY_BANDS.keys())[band_idx]}")
# ax[j, i].axvline(idx_best, color='green', linestyle='--', label='Best fold')
# ax[j, i].axvline(1.5*sr_decimated, color='red', linestyle='--', label='fb')
# ax[j, i].legend()
# diff.append(to_plot[:, idx_best])
        # t, p = ttest_ind(diff[0], diff[1])
#     ax[j, -1].boxplot(diff)
#     ax[j, -1].set_title(f"T-test p-value: {p:.3e}")
# plt.tight_layout()


In [ ]:
t_, p = ttest_ind(pos_sig, neg_sig)